In [ ]:
import pandas as pd
import numpy as np

# 🔹 Load Excel file
file_path = "PLAN_ACTUAL.xlsx"   # change to your file name
df = pd.read_excel(file_path)

# 🔹 Separate material column
materials = df['Material']

# 🔹 Take only demand columns (all except Material)
demand_data = df.drop(columns=['Material'])

# 🔹 Calculate statistics row-wise
mean_demand = demand_data.mean(axis=1)
std_dev = demand_data.std(axis=1, ddof=1)
count = demand_data.count(axis=1)

# 🔹 Standard Error
std_error = std_dev / np.sqrt(count)

# 🔹 95% Confidence Interval
z = 1.96
lower_ci = mean_demand - z * std_error
upper_ci = mean_demand + z * std_error

# 🔹 Create result dataframe
result = pd.DataFrame({
    'Material': materials,
    'Mean_Demand': mean_demand,
    'Std_Dev': std_dev,
    'Lower_95_CI': lower_ci,
    'Upper_95_CI': upper_ci
})

print(result)

# 🔹 Optional: Save to Excel
result.to_excel("Confidence_Interval_Output.xlsx", index=False)


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# 🔹 Load Excel file
file_path = "PLAN_ACTUAL.xlsx"   # change to your file name
df = pd.read_excel(file_path)

# 🔹 Separate material column
materials = df['Material']

# 🔹 Take only demand columns (all except Material)
demand_data = df.drop(columns=['Material'])

# 🔹 Calculate statistics row-wise
mean_demand = demand_data.mean(axis=1)
std_dev = demand_data.std(axis=1, ddof=1)
count = demand_data.count(axis=1)

# 🔹 Standard Error
std_error = std_dev / np.sqrt(count)

# 🔹 95% Confidence Interval using t-score
alpha = 0.05
# t critical value depends on df = count - 1
t_values = stats.t.ppf(1 - alpha/2, df=count - 1)

lower_ci = mean_demand - t_values * std_error
upper_ci = mean_demand + t_values * std_error

# 🔹 Create result dataframe
result = pd.DataFrame({
    'Material': materials,
    'Mean_Demand': mean_demand,
    'Std_Dev': std_dev,
    'Lower_95_CI': lower_ci,
    'Upper_95_CI': upper_ci
})

print(result)

# 🔹 Optional: Save to Excel
result.to_excel("Confidence_Interval_Output.xlsx", index=False)


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm

# =====================================
# CONFIGURATION
# =====================================

ROLLING_DAYS = 7
SERVICE_LEVEL = 0.95
RAMP_FACTOR = 1.0

# =====================================
# LOAD FILE
# =====================================

df = pd.read_excel("plan_actual.xlsx")

# Clean column names
df.columns = df.columns.str.replace(" Total Production Plan", "", regex=False)

# =====================================
# WIDE → LONG
# =====================================

df_long = df.melt(
    id_vars=["Material"],
    var_name="Date",
    value_name="Actual_Demand"
)

df_long["Date"] = pd.to_datetime(df_long["Date"])
df_long = df_long.sort_values(["Material", "Date"]).reset_index(drop=True)

# =====================================
# CALCULATE ROLLING STATS
# =====================================

df_long["Mean_Demand"] = (
    df_long.groupby("Material")["Actual_Demand"]
    .rolling(ROLLING_DAYS)
    .mean()
    .reset_index(level=0, drop=True)
)

df_long["Std_Demand"] = (
    df_long.groupby("Material")["Actual_Demand"]
    .rolling(ROLLING_DAYS)
    .std()
    .reset_index(level=0, drop=True)
)

df_long["Std_Demand"] = df_long["Std_Demand"].fillna(0)
df_long["Mean_Demand"] = df_long["Mean_Demand"].fillna(0)

# =====================================
# SAFE PRODUCTION
# =====================================

Z = norm.ppf(SERVICE_LEVEL)

df_long["Safe_Production"] = (
    df_long["Mean_Demand"] + Z * df_long["Std_Demand"]
)

# =====================================
# DYNAMIC RAMP — CORRECT IMPLEMENTATION
# =====================================

final_list = []

for material, group in df_long.groupby("Material"):

    group = group.sort_values("Date").copy()
    group["Final_Production"] = 0

    prev_prod = None

    for i in range(len(group)):

        safe = group.iloc[i]["Safe_Production"]
        sigma = group.iloc[i]["Std_Demand"]

        if prev_prod is None:
            final = safe
        else:
            ramp = RAMP_FACTOR * sigma
            upper = prev_prod + ramp
            lower = prev_prod - ramp

            final = min(safe, upper)
            final = max(final, lower)
            final = max(final, 0)

        group.iloc[i, group.columns.get_loc("Final_Production")] = final
        prev_prod = final

    final_list.append(group)

# Combine all materials back
df_final = pd.concat(final_list).reset_index(drop=True)

# =====================================
# SAVE OUTPUT
# =====================================

df_final.to_excel("dynamic_stable_production.xlsx", index=False)

print("Correct long-format production plan generated.")
